# VASP Refiner V3 Preset Training - Gemma 4 E2B Unsloth QLoRA

This notebook fine-tunes Gemma 4 E2B on the Refiner V3 professional preset examples:

`vasp/a2v/finetuning/refiner_v3_presets/refiner_v3_preset_150.jsonl`

It teaches the refiner to output professional preset names like `cinematic_history`, `science_discovery`, `caption_bottom_safe`, `visual_center_800h`, etc., instead of inventing large raw style objects.

The notebook can either:

- train a fresh LoRA adapter from the base Gemma 4 E2B model, or
- continue from an existing refiner adapter zip in Drive if `USE_PREVIOUS_ADAPTER = True`.

Output zip saved to Drive:

`/content/drive/MyDrive/refiner_v3_preset_e2b_unsloth_qlora_v1.zip`


In [1]:
# 1) Check GPU runtime
!nvidia-smi


Thu May 14 22:41:53 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX PRO 6000 Blac...    Off |   00000000:05:00.0 Off |                    0 |
| N/A   29C    P0             48W /  600W |       0MiB /  97887MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [2]:
# 2) Clone/pull repo
REPO_URL = "https://github.com/theextraordinary/VASP.git"
REPO_DIR = "/content/VASP"
import os
if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    %cd {REPO_DIR}
    !git fetch origin
    !git reset --hard origin/main
%cd {REPO_DIR}


Cloning into '/content/VASP'...
remote: Enumerating objects: 835, done.
remote: Counting objects: 100% (835/835), done.
remote: Compressing objects: 100% (649/649), done.
remote: Total 835 (delta 216), reused 794 (delta 177), pack-reused 0 (from 0)
Receiving objects: 100% (835/835), 7.26 MiB | 28.71 MiB/s, done.
Resolving deltas: 100% (216/216), done.
/content/VASP


In [3]:
# 3) Install dependencies (restart runtime if Colab upgrades numpy/pyarrow mid-session)
!pip -q install -U unsloth datasets trl peft accelerate bitsandbytes huggingface_hub

print("If you see a numpy/pyarrow binary incompatibility error after this cell, restart runtime and continue from cell 4.")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.0/56.0 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.4/67.4 MB 38.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 64.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 61.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 26.3 MB/s eta 0:00:0000:01:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 661.5/661.5 kB 85.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 55.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 192.5 MB/s eta 0:00:00 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 428.0/428.0 kB 58.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 152.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 78.7 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 29.3 MB/s eta 0:00

In [4]:
# 4) Hugging Face login (needed for gated Gemma checkpoints)
from huggingface_hub import login
login()


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


In [5]:
# 5) Config
from pathlib import Path

MODEL_NAME = "google/gemma-4-e2b-it"
DATASET_PATH = Path("vasp/a2v/finetuning/refiner_v3_presets/refiner_v3_preset_150.jsonl")

# Optional continued-training path. Set USE_PREVIOUS_ADAPTER=True if you already have a refiner adapter zip.
USE_PREVIOUS_ADAPTER = False
PREVIOUS_ADAPTER_ZIP = Path("/content/drive/MyDrive/refiner_v3_e2b_unsloth_qlora.zip")
PREVIOUS_ADAPTER_UNZIP_ROOT = Path("/content/adapters/refiner_v3_previous")

RUN_NAME = "refiner_v3_preset_e2b_unsloth_qlora_v1"
OUTPUT_DIR = Path("vasp/a2v/finetuning/refiner_v3_presets/output") / RUN_NAME
ZIP_NAME = RUN_NAME + ".zip"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MAX_SEQ_LENGTH = 4096
SEED = 42
FULL_TRAIN = True
VAL_RATIO = 0.0
NUM_TRAIN_EPOCHS = 6
LEARNING_RATE = 3e-5


In [6]:
# 6) Mount Drive and prepare dataset
from google.colab import drive
from pathlib import Path
import shutil, json

drive.mount('/content/drive')

if not DATASET_PATH.exists():
    print("Dataset missing; generating examples now...")
    !python -m vasp.a2v.finetuning.refiner_v3_presets.create_refiner_preset_examples

print("Validating dataset...")
!python -m vasp.a2v.finetuning.refiner_v3_presets.validate_refiner_preset_examples

if not DATASET_PATH.exists():
    raise FileNotFoundError(f"Missing dataset: {DATASET_PATH}")

if USE_PREVIOUS_ADAPTER:
    if not PREVIOUS_ADAPTER_ZIP.exists():
        raise FileNotFoundError(f"Missing previous adapter zip: {PREVIOUS_ADAPTER_ZIP}")
    if PREVIOUS_ADAPTER_UNZIP_ROOT.exists():
        shutil.rmtree(PREVIOUS_ADAPTER_UNZIP_ROOT)
    PREVIOUS_ADAPTER_UNZIP_ROOT.mkdir(parents=True, exist_ok=True)
    !unzip -q -o "{PREVIOUS_ADAPTER_ZIP}" -d "{PREVIOUS_ADAPTER_UNZIP_ROOT}"

    def find_adapter_dir(root: Path) -> Path:
        candidates = sorted(root.rglob("adapter_config.json"))
        if not candidates:
            raise FileNotFoundError(f"No adapter_config.json found after unzipping {PREVIOUS_ADAPTER_ZIP}")
        for cfg in candidates:
            parent = cfg.parent
            if (parent / "adapter_model.safetensors").exists() or (parent / "adapter_model.bin").exists():
                return parent
        return candidates[0].parent

    PREVIOUS_ADAPTER_DIR = find_adapter_dir(PREVIOUS_ADAPTER_UNZIP_ROOT)
    print("Continuing from previous adapter:", PREVIOUS_ADAPTER_DIR)
else:
    PREVIOUS_ADAPTER_DIR = None
    print("Training fresh LoRA adapter from base model.")

print("Dataset:", DATASET_PATH)
print("Output:", OUTPUT_DIR)


Mounted at /content/drive
Validating dataset...
{
  "ok": true,
  "errors": [],
  "error_count": 0
}
Training fresh LoRA adapter from base model.
Dataset: vasp/a2v/finetuning/refiner_v3_presets/refiner_v3_preset_150.jsonl
Output: vasp/a2v/finetuning/refiner_v3_presets/output/refiner_v3_preset_e2b_unsloth_qlora_v1


In [7]:
# 7) Load dataset
import json, random
from datasets import Dataset

random.seed(SEED)
rows = [json.loads(x) for x in DATASET_PATH.read_text(encoding="utf-8").splitlines() if x.strip()]
random.shuffle(rows)

if FULL_TRAIN:
    train_rows = rows
    val_rows = []
else:
    val_n = max(5, int(len(rows) * VAL_RATIO))
    val_rows = rows[:val_n]
    train_rows = rows[val_n:]

train_ds = Dataset.from_list(train_rows)
val_ds = Dataset.from_list(val_rows) if val_rows else None
print(f"train={len(train_ds)} val={len(val_ds) if val_ds is not None else 0} full_train={FULL_TRAIN}")


train=150 val=0 full_train=True


In [8]:
# 8) Load model + tokenizer with Unsloth
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
)

if getattr(tokenizer, "pad_token", None) is None and hasattr(tokenizer, "eos_token"):
    tokenizer.pad_token = tokenizer.eos_token
if hasattr(tokenizer, "padding_side"):
    tokenizer.padding_side = "left"


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.5.2: Fast Gemma4 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA RTX PRO 6000 Blackwell Server Edition. Num GPUs = 1. Max memory: 94.971 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 12.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/2011 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/203 [00:00<?, ?B/s]

processor_config.json: 0.00B [00:00, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/32.2M [00:00<?, ?B/s]

In [9]:
# 9) Convert chat rows to text via tokenizer chat template
def to_text(example):
    text = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False,
    )
    return {"text": text}

train_ds = train_ds.map(to_text)
if val_ds is not None:
    val_ds = val_ds.map(to_text)
print(train_ds[0]["text"][:1200])


Map:   0%|          | 0/150 [00:00<?, ? examples/s]

<bos><|turn>user
You are Refiner V3. Choose professional preset names.

CREATIVITY LEVEL: 4

TARGET BUNDLE HINT: comedy_reaction

SEGMENT:
```json
{
  "segment_id": "segment_064",
  "matched_text": "everyone was completely shocked",
  "t_start": 153.6,
  "t_end": 155.8,
  "media_id": "media_64",
  "media": {
    "element_id": "media_64",
    "type": "video",
    "source_path": "assets/fake/media_64.mp4",
    "about": "everyone was completely shocked",
    "aim": "show during 'everyone was completely shocked'"
  },
  "caption_groups": [
    {
      "index": 64,
      "text": "everyone was completely shocked",
      "start": 153.6,
      "end": 155.8
    }
  ],
  "warnings": []
}
```<turn|>
<|turn>model
{"segment_id": "segment_064", "creativity_level": 4, "preset_bundle": "comedy_reaction", "background_preset": "comedy_pop_pink", "caption_preset": "meme_bold_pop", "caption_layout_preset": "caption_bottom_safe", "visual_layout_preset": "visual_square_center_card", "caption_animation_prese

In [10]:
# 10) Prepare LoRA adapter
from peft import PeftModel


def unwrap_gemma4_clippable_linear(module, prefix=""):
    """PEFT cannot inject LoRA into Gemma4ClippableLinear wrappers.

    Replace wrapper modules with their inner .linear layer so PEFT sees supported torch.nn.Linear modules.
    """
    replaced = []
    for name, child in list(module.named_children()):
        child_path = f"{prefix}.{name}" if prefix else name
        if child.__class__.__name__ == "Gemma4ClippableLinear" and hasattr(child, "linear"):
            setattr(module, name, child.linear)
            replaced.append(child_path)
            continue
        replaced.extend(unwrap_gemma4_clippable_linear(child, child_path))
    return replaced

unwrapped_modules = unwrap_gemma4_clippable_linear(model)
print(f"Unwrapped Gemma4ClippableLinear modules for PEFT: {len(unwrapped_modules)}")
if unwrapped_modules:
    print("First unwrapped modules:", unwrapped_modules[:12])

if USE_PREVIOUS_ADAPTER and PREVIOUS_ADAPTER_DIR is not None:
    model = PeftModel.from_pretrained(
        model,
        str(PREVIOUS_ADAPTER_DIR),
        adapter_name="refiner_v3",
        is_trainable=True,
    )
    print("Continuing training from:", PREVIOUS_ADAPTER_DIR)
else:
    model = FastLanguageModel.get_peft_model(
        model,
        r=16,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        lora_alpha=32,
        lora_dropout=0.05,
        bias="none",
        use_gradient_checkpointing="unsloth",
        random_state=SEED,
        use_rslora=False,
        loftq_config=None,
    )
    print("Created fresh LoRA adapter.")

model.train()
model.print_trainable_parameters()


Unwrapped Gemma4ClippableLinear modules for PEFT: 232
First unwrapped modules: ['model.vision_tower.encoder.layers.0.self_attn.q_proj', 'model.vision_tower.encoder.layers.0.self_attn.k_proj', 'model.vision_tower.encoder.layers.0.self_attn.v_proj', 'model.vision_tower.encoder.layers.0.self_attn.o_proj', 'model.vision_tower.encoder.layers.0.mlp.gate_proj', 'model.vision_tower.encoder.layers.0.mlp.up_proj', 'model.vision_tower.encoder.layers.0.mlp.down_proj', 'model.vision_tower.encoder.layers.1.self_attn.q_proj', 'model.vision_tower.encoder.layers.1.self_attn.k_proj', 'model.vision_tower.encoder.layers.1.self_attn.v_proj', 'model.vision_tower.encoder.layers.1.self_attn.o_proj', 'model.vision_tower.encoder.layers.1.mlp.gate_proj']


Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.


Created fresh LoRA adapter.
trainable params: 31,039,488 || all params: 5,154,217,504 || trainable%: 0.6022


In [11]:
# 11) Train with TRL SFTTrainer
from trl import SFTTrainer
from transformers import TrainingArguments
import inspect

_ta_params = inspect.signature(TrainingArguments.__init__).parameters
_strategy_key = "evaluation_strategy" if "evaluation_strategy" in _ta_params else "eval_strategy"

train_kwargs = dict(
    output_dir=str(OUTPUT_DIR / "checkpoints"),
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    num_train_epochs=NUM_TRAIN_EPOCHS,
    learning_rate=LEARNING_RATE,
    warmup_ratio=0.03,
    weight_decay=0.0,
    logging_steps=5,
    save_steps=25,
    save_strategy="steps",
    bf16=True,
    fp16=False,
    optim="adamw_8bit",
    lr_scheduler_type="cosine",
    seed=SEED,
    report_to="none",
)
if val_ds is not None:
    train_kwargs.update(
        per_device_eval_batch_size=2,
        eval_steps=25,
        load_best_model_at_end=True,
    )
    train_kwargs[_strategy_key] = "steps"
else:
    train_kwargs[_strategy_key] = "no"
    train_kwargs["load_best_model_at_end"] = False

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    packing=False,
    args=TrainingArguments(**train_kwargs),
)

train_result = trainer.train()
print(train_result)


warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Unsloth: Tokenizing ["text"] (num_proc=52):   0%|          | 0/150 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': 2}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 150 | Num Epochs = 6 | Total steps = 60
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 8 x 1) = 16
 "-____-"     Trainable parameters = 31,039,488 of 5,154,217,504 (0.60% trained)


Step,Training Loss
5,0.327110
10,0.361175
15,0.213526
20,0.238684
25,0.164145
30,0.198680
35,0.139750
40,0.175335
45,0.126846
50,0.164299


Unsloth: Restored added_tokens_decoder metadata in vasp/a2v/finetuning/refiner_v3_presets/output/refiner_v3_preset_e2b_unsloth_qlora_v1/checkpoints/checkpoint-25/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in vasp/a2v/finetuning/refiner_v3_presets/output/refiner_v3_preset_e2b_unsloth_qlora_v1/checkpoints/checkpoint-50/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in vasp/a2v/finetuning/refiner_v3_presets/output/refiner_v3_preset_e2b_unsloth_qlora_v1/checkpoints/checkpoint-60/tokenizer_config.json.


TrainOutput(global_step=60, training_loss=0.19948336879412334, metrics={'train_runtime': 103.8479, 'train_samples_per_second': 8.667, 'train_steps_per_second': 0.578, 'total_flos': 7404125141585280.0, 'train_loss': 0.19948336879412334, 'epoch': 6.0})


In [12]:
# 12) Save adapter + tokenizer
adapter_dir = OUTPUT_DIR / "adapter"
trainer.model.save_pretrained(str(adapter_dir))
tokenizer.save_pretrained(str(adapter_dir))
print("Saved adapter:", adapter_dir)


Unsloth: Restored added_tokens_decoder metadata in vasp/a2v/finetuning/refiner_v3_presets/output/refiner_v3_preset_e2b_unsloth_qlora_v1/adapter/tokenizer_config.json.


Saved adapter: vasp/a2v/finetuning/refiner_v3_presets/output/refiner_v3_preset_e2b_unsloth_qlora_v1/adapter


In [13]:
# 13) Zip adapter
!cd /content/VASP && rm -f {ZIP_NAME} && zip -r {ZIP_NAME} {adapter_dir}


  adding: vasp/a2v/finetuning/refiner_v3_presets/output/refiner_v3_preset_e2b_unsloth_qlora_v1/adapter/ (stored 0%)
  adding: vasp/a2v/finetuning/refiner_v3_presets/output/refiner_v3_preset_e2b_unsloth_qlora_v1/adapter/adapter_model.safetensors (deflated 23%)
  adding: vasp/a2v/finetuning/refiner_v3_presets/output/refiner_v3_preset_e2b_unsloth_qlora_v1/adapter/adapter_config.json (deflated 58%)
  adding: vasp/a2v/finetuning/refiner_v3_presets/output/refiner_v3_preset_e2b_unsloth_qlora_v1/adapter/README.md (deflated 65%)
  adding: vasp/a2v/finetuning/refiner_v3_presets/output/refiner_v3_preset_e2b_unsloth_qlora_v1/adapter/tokenizer.json (deflated 83%)
  adding: vasp/a2v/finetuning/refiner_v3_presets/output/refiner_v3_preset_e2b_unsloth_qlora_v1/adapter/chat_template.jinja (deflated 82%)
  adding: vasp/a2v/finetuning/refiner_v3_presets/output/refiner_v3_preset_e2b_unsloth_qlora_v1/adapter/processor_config.json (deflated 69%)
  adding: vasp/a2v/finetuning/refiner_v3_presets/output/refiner

In [14]:
# 14) Save zip to Google Drive
from google.colab import drive
drive.mount('/content/drive')
!cp /content/VASP/{ZIP_NAME} /content/drive/MyDrive/
print("Saved adapter zip to Drive:", "/content/drive/MyDrive/" + ZIP_NAME)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Saved adapter zip to Drive: /content/drive/MyDrive/refiner_v3_preset_e2b_unsloth_qlora_v1.zip


In [ ]:
# 15) Optional quick inference sanity check
from unsloth import FastLanguageModel
FastLanguageModel.for_inference(model)

sample = rows[0]["messages"][0]["content"]
messages = [{"role": "user", "content": sample}]
prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
outputs = model.generate(**inputs, max_new_tokens=700, temperature=0.2, do_sample=True)
print(tokenizer.decode(outputs[0], skip_special_tokens=True)[-2500:])


In [ ]:
# 16) Download zip to local machine
from google.colab import files
files.download("/content/VASP/" + ZIP_NAME)
